In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import torch
import time

# VRAM 용량
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# CUDA 연산 속도 테스트
a = torch.randn(10000, 10000, device="cuda")
b = torch.randn(10000, 10000, device="cuda")

torch.cuda.synchronize()
start = time.time()
c = a @ b
torch.cuda.synchronize()
print(f"10000x10000 행렬곱: {time.time() - start:.3f}초")

In [ ]:
import torch
import torch.nn as nn
import time

# 간단한 CNN으로 GPU 학습 속도 테스트
model = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(64, 128, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(128, 10)
).cuda()

optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

# 가짜 데이터로 100 step 학습
torch.cuda.synchronize()
start = time.time()
for i in range(100):
    x = torch.randn(64, 3, 32, 32, device="cuda")
    y = torch.randint(0, 10, (64,), device="cuda")
    loss = loss_fn(model(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()

print(f"CNN 100 step 학습: {time.time() - start:.2f}초")
print(f"VRAM 사용량: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")

# Stable Diffusion Quick Test
Run the cells below in order to test Stable Diffusion on this GPU.

In [ ]:
!pip install diffusers transformers accelerate safetensors

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

pipe.enable_attention_slicing()
print(f"VRAM usage: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
print("Ready!")

In [ ]:
prompt = "a cat astronaut riding a horse on mars, digital art, highly detailed"

image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
).images[0]

image